# 02 — Le tre famiglie, la combinazione e il modello "alla NewsGuard" (tentativo 09)

Ogni testata è **una riga**. Tre famiglie di ingresso, tutte osservabili per qualunque sito:

| famiglia | che cosa entra |
|---|---|
| **testo** | la media dei vettori TF-IDF dei suoi articoli (TF-IDF 1–2 grammi come nelle Versioni 2–4, stimato sul solo training) |
| **link** | i domini citati (un TF-IDF sui domini, almeno 3 testate del training che li citano), le quote per tipo (social, video, video alternativi, istituzionali, Wikipedia, testate NewsGuard) e l'**affidabilità dei siti citati**: punteggio medio delle testate citate e quota di citazioni verso testate inaffidabili, calcolati **solo con le etichette delle testate di training** (mai del test, mai dello stesso gruppo) |
| **pagina** | servizi di terze parti per categoria, reti pubblicitarie, pagine di trasparenza, firma, data, dati strutturati, opinione dichiarata, paywall, commenti, cookie, contenuti incorporati, dimensioni |

I criteri e il punteggio NewsGuard sono solo **obiettivi**, mai ingressi.

**I modelli** (tutti regressione logistica, pesi per testata come nelle Versioni 2–4: un gruppo vale al più 5 testate,
classi bilanciate sui pesi):

1. **un modello per famiglia** (5 livelli), con C scelto in una validazione interna per gruppi (3 fold) sul solo
   training, con la F1 macro delle testate valutate;
2. **la combinazione** (*stacking*): un modello che impara a pesare le probabilità delle famiglie, addestrato sulle
   probabilità *fuori fold* della validazione interna; anche per coppie di famiglie (ablazione);
3. **"alla NewsGuard"**: 9 classificatori sì/no, uno per criterio, ognuno combinando le tre famiglie come sopra; il
   criterio è rispettato se la probabilità è ≥ 0,5; il punteggio è la somma dei pesi NewsGuard dei criteri
   rispettati; dal punteggio il livello.

Valutazione: le 301 testate valutate, fold per gruppo (reti ∪ owner ∪ Parent Domain). Si salva tutto: probabilità
per testata di ogni modello, criteri stimati, C scelti, coefficienti dei modelli link e pagina.

In [ ]:
import json
import re
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfTransformer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

from comune import (CRITERI, PUNTI, RESULTS, SEED, SEGNALI, VERSIONI, livello_da_punteggio, metriche_di, pesi_bilanciati,
                    registratore, testate)

In [ ]:
C_GRIGLIA = "0.01,0.1,1,10"
FAMIGLIE = "testo,link,pagina"
CRITERI_SI = True           # anche il modello "alla NewsGuard"
PROVA = False               # True: 2 fold, griglia ridotta

In [ ]:
OUT = RESULTS / ("prova" if PROVA else "") / "modelli"
OUT.mkdir(parents=True, exist_ok=True)
registra = registratore(OUT / "avanzamento.log")
T0 = time.time()
GRIGLIA = [float(c) for c in C_GRIGLIA.split(",")]
FAM = FAMIGLIE.split(",")
t = testate()
S = pd.read_csv(SEGNALI / "testate.csv.gz", index_col=0).reindex(t.index)
lnk = pd.read_csv(SEGNALI / "link_citati.csv.gz")
lnk = lnk[lnk.citato.map(t.gruppo).ne(lnk.domain.map(t.gruppo))]          # via le citazioni dentro il gruppo
art = pd.read_csv(VERSIONI / "dataset_finale.csv", usecols=["domain", "text"]); art["text"] = art.text.fillna("")
idx = {d: i for i, d in enumerate(t.index)}
A = sp.csr_matrix((np.ones(len(art)), (art.domain.map(idx).to_numpy(), np.arange(len(art)))), shape=(len(t), len(art)))
A = sp.diags(1 / np.maximum(A.sum(1).A1, 1)) @ A                          # media per testata
COL_LINK_TIPI = ["link citati per articolo"] + [c for c in S.columns if c.startswith("link: ")]
COL_PAGINA = [c for c in S.columns if c not in COL_LINK_TIPI and c not in ("link citati nel testo", "parole del testo estratto")]
y = t.label.to_numpy(); peso = t.peso.to_numpy(); val = t.valutazione.to_numpy(); folds = sorted(t.fold_gruppo.unique())
if PROVA:
    folds, GRIGLIA = folds[:2], GRIGLIA[1:3]
registra(f"=== {len(t)} testate ({val.sum()} valutate), {len(art)} articoli, {len(lnk)} citazioni; famiglie {FAM}; C {GRIGLIA} ===")

## Le matrici delle famiglie, dato un insieme di testate di training

Tutto ciò che si stima dai dati (vocabolario e IDF del testo e dei domini, standardizzazione, affidabilità dei siti
citati) si stima **solo** sulle testate di training passate alla funzione.

In [ ]:
def matrici(tr):
    tr = np.asarray(tr)
    M = {}
    if "testo" in FAM:
        v = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9, sublinear_tf=True, max_features=100_000, strip_accents="unicode")
        art_tr = art.domain.map(idx).isin(tr).to_numpy()
        v.fit(art.text[art_tr])
        X = A @ v.transform(art.text)
        M["testo"] = (sp.diags(1 / np.maximum(sp.linalg.norm(X, axis=1), 1e-12)) @ X).tocsr()
    if "link" in FAM:
        cnt = lnk.groupby(["domain", "citato"]).volte.sum()
        presenti = set(cnt.index.get_level_values(0))
        righe = [cnt.loc[d].to_dict() if d in presenti else {} for d in t.index]
        dv = DictVectorizer(); D = dv.fit_transform(righe)
        tenuti = np.asarray((D[tr] > 0).sum(0)).ravel() >= 3                   # domini citati da almeno 3 testate del training
        D = D[:, tenuti]
        tf = TfidfTransformer(sublinear_tf=True).fit(D[tr])
        Dt = tf.transform(D)
        # affidabilità dei siti citati: solo testate di training (e mai dello stesso gruppo, già tolte)
        noto = set(t.index[tr])
        c = lnk[lnk.citato.isin(noto)]
        sc = c.citato.map(t.score)
        w = c.volte
        media = (sc * w).groupby(c.domain).sum() / w.groupby(c.domain).sum()
        inaff = ((sc < 60) * w).groupby(c.domain).sum() / w.groupby(c.domain).sum()
        n = w.groupby(c.domain).sum()
        V = pd.DataFrame({"punteggio medio dei siti citati": media, "quota citazioni verso inaffidabili": inaff,
                          "citazioni di testate note (log)": np.log1p(n)}).reindex(t.index)
        V["nessuna citazione di testate note"] = V["punteggio medio dei siti citati"].isna().astype(float)
        dense = pd.concat([S[COL_LINK_TIPI], V], axis=1)
        dense = dense.fillna(dense.iloc[tr].mean()).fillna(0)
        sc_ = StandardScaler().fit(dense.iloc[tr])
        M["link"] = sp.hstack([Dt, sp.csr_matrix(sc_.transform(dense))]).tocsr()
        M["_nomi_link"] = list(np.asarray(dv.get_feature_names_out())[tenuti]) + list(dense.columns)
    if "pagina" in FAM:
        P = S[COL_PAGINA].fillna(S[COL_PAGINA].iloc[tr].mean()).fillna(0)
        sc_ = StandardScaler().fit(P.iloc[tr])
        M["pagina"] = sp.csr_matrix(sc_.transform(P))
        M["_nomi_pagina"] = list(P.columns)
    return M

## Il modello di una famiglia, e la validazione interna

In [ ]:
def adatta(X, yy, ww, c, binario=False):
    pw = pesi_bilanciati(yy, ww)
    return LogisticRegression(C=c, max_iter=5000, random_state=SEED).fit(X, yy, sample_weight=pw)


def proba(m, X, k):
    P = np.zeros((X.shape[0], k)); P[:, m.classes_.astype(int)] = m.predict_proba(X)
    return P


def punteggio(yv, Pv, binario):
    if binario:
        return balanced_accuracy_score(yv, (Pv[:, 1] >= 0.5).astype(int))
    return f1_score(yv, Pv.argmax(1), average="macro", labels=range(5), zero_division=0)


def interni(tr):
    # 3 fold per gruppi sulle testate di training: (indici di training, indici di validazione, matrici)
    tt = t.iloc[tr]
    out = []
    for a, b in StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(tt, tt.label, tt.gruppo):
        out.append((tr[a], tr[b], matrici(tr[a])))
    return out


def famiglia(fam, tr, te, M, INT, yy, k, binario):
    # sceglie C nella validazione interna; ritorna probabilità sul test e fuori fold sul training
    noti = ~np.isnan(yy)
    punti, oof = {}, {}
    for c in GRIGLIA:
        s, P = [], np.full((len(t), k), np.nan)
        for a, b, Mi in INT:
            a, b2 = a[noti[a]], b[noti[b]]
            m = adatta(Mi[fam][a], yy[a].astype(int), peso[a], c, binario)
            P[b2] = proba(m, Mi[fam][b2], k)
            bv = b2[val[b2]]
            s.append(punteggio(yy[bv].astype(int), P[bv], binario))
        punti[c], oof[c] = float(np.mean(s)), P
    C = max(punti, key=punti.get)
    tr2 = tr[noti[tr]]
    m = adatta(M[fam][tr2], yy[tr2].astype(int), peso[tr2], C, binario)
    return proba(m, M[fam][te], k), oof[C][tr], C, punti, m

## Tutti i fold: famiglie, combinazioni, criteri

In [ ]:
COMBO = {"combinato (tutte)": FAM}
if len(FAM) == 3:
    COMBO.update({f"combinato ({a}+{b})": [a, b] for a, b in [("testo", "link"), ("testo", "pagina"), ("link", "pagina")]})
PROB = defaultdict(lambda: np.full((len(t), 5), np.nan))
CRIT = {c: np.full(len(t), np.nan) for c in CRITERI}
scelte, coef = [], []


def combina(oof_fam, te_fam, yy, tr, k):
    # meta-modello sulle probabilità fuori fold delle famiglie (log), addestrato sul solo training
    noti = ~np.isnan(yy[tr])
    Z = np.hstack([np.log(np.clip(oof_fam[f], 1e-6, 1)) for f in oof_fam])[noti]
    Zt = np.hstack([np.log(np.clip(te_fam[f], 1e-6, 1)) for f in te_fam])
    m = adatta(Z, yy[tr][noti].astype(int), peso[tr][noti], 1.0)
    return proba(m, Zt, k)


for f in folds:
    t1 = time.time()
    tr, te = np.where(t.fold_gruppo != f)[0], np.where(t.fold_gruppo == f)[0]
    M = matrici(tr); INT = interni(tr)
    registra(f"fold {f}: matrici pronte ({(time.time() - t1) / 60:.1f} min)")
    # 5 livelli
    te_p, oof_p = {}, {}
    for fam in FAM:
        te_p[fam], oof_p[fam], C, punti, m = famiglia(fam, tr, te, M, INT, y.astype(float), 5, False)
        PROB[fam][te] = te_p[fam]
        scelte.append({"fold": f, "obiettivo": "livello", "famiglia": fam, "C": C, **{f"F1 interna C={k}": v for k, v in punti.items()}})
        if fam in ("link", "pagina"):
            nomi = M[f"_nomi_{fam}"]
            for k_ in range(m.coef_.shape[0]):
                for j in np.argsort(m.coef_[k_])[::-1][:15]:
                    coef.append({"fold": f, "famiglia": fam, "livello": f"l{int(m.classes_[k_]) + 1}", "segnale": nomi[j], "peso": m.coef_[k_, j]})
        mm = metriche_di(y[te][val[te]], te_p[fam][val[te]].argmax(1))
        registra(f"fold {f} {fam}: C {C}, F1 {mm['f1_macro']:.3f}, binario {mm['bin_accuratezza']:.3f}")
    for nome, fs in COMBO.items():
        PROB[nome][te] = combina({x: oof_p[x] for x in fs}, {x: te_p[x] for x in fs}, y.astype(float), tr, 5)
    mm = metriche_di(y[te][val[te]], PROB["combinato (tutte)"][te][val[te]].argmax(1))
    registra(f"fold {f} combinato: F1 {mm['f1_macro']:.3f}, binario {mm['bin_accuratezza']:.3f}")
    # alla NewsGuard: un classificatore sì/no per criterio, che combina le famiglie
    if CRITERI_SI:
        for c in CRITERI:
            yc = t[c].to_numpy()
            te_c, oof_c = {}, {}
            for fam in FAM:
                te_c[fam], oof_c[fam], C, punti, _ = famiglia(fam, tr, te, M, INT, yc, 2, True)
                scelte.append({"fold": f, "obiettivo": c, "famiglia": fam, "C": C})
            CRIT[c][te] = combina(oof_c, te_c, yc, tr, 2)[:, 1]
        registra(f"fold {f}: criteri fatti ({(time.time() - t1) / 60:.1f} min dal via del fold)")

## Il punteggio "alla NewsGuard" e il salvataggio

In [ ]:
if CRITERI_SI:
    SI = np.column_stack([CRIT[c] >= 0.5 for c in CRITERI]).astype(float)
    punteggio_stimato = SI @ np.array(PUNTI)
    liv = livello_da_punteggio(punteggio_stimato)
    PROB["alla NewsGuard"] = np.eye(5)[np.nan_to_num(liv, nan=0).astype(int)]
    PROB["alla NewsGuard"][np.isnan(CRIT[CRITERI[0]])] = np.nan
fatti = ~np.isnan(PROB[FAM[0]][:, 0])
righe = pd.DataFrame({"domain": t.index, "fold": t.fold_gruppo, "valutazione": val, "y_true": y, "score": t.score})
for nome, P in PROB.items():
    for k in range(5):
        righe[f"{nome} | p_l{k + 1}"] = P[:, k]
if CRITERI_SI:
    for c in CRITERI:
        righe[f"criterio stimato | {c}"] = CRIT[c]
    righe["punteggio stimato"] = punteggio_stimato
righe[fatti].to_csv(OUT / "predizioni_testate.csv", index=False)
pd.DataFrame(scelte).to_csv(OUT / "scelte_C.csv", index=False)
pd.DataFrame(coef).to_csv(OUT / "coefficienti_link_pagina.csv", index=False)
v = fatti & val
tab = {nome: {k: m for k, m in metriche_di(y[v], P[v].argmax(1)).items() if not isinstance(m, list)} for nome, P in PROB.items()}
tab = pd.DataFrame(tab).T[["n", "f1_macro", "accuratezza", "tolerant_1", "bin_accuratezza", "bin_f1_macro"]]
tab.to_csv(OUT / "metriche.csv")
if CRITERI_SI:
    cr = pd.DataFrame({c: {"accuratezza bilanciata": balanced_accuracy_score(t[c][v].dropna(), (CRIT[c][v] >= 0.5)[t[c][v].notna().to_numpy()]),
                           "quota sì (vera)": t[c][v].mean()} for c in CRITERI}).T
    cr.to_csv(OUT / "criteri.csv"); display(cr.round(3))
(OUT / "FATTO").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))
registra(f"=== finito in {(time.time() - T0) / 60:.1f} min ===\n" + tab.round(3).to_string())
tab.round(3)